# XPS fit validation — this code vs KherveFitting (one-time check)

Follow **XPS_validation_checklist.md** alongside this notebook.

1. **Section 1** exports the validation spectra (already charge-corrected) and this code's results.
2. Fit those files in KherveFitting with the settings in the checklist; type its results into `kherve_results.csv`.
3. **Section 2** compares both and prints PASS / CHECK for every number.

In [ ]:
import numpy as np
import pandas as pd
from battery_analysis.project import Project
from battery_analysis.analysis.xps import fitting as xf, sources

project = Project("..")
project.use("xps", "xps.yaml")
BATCH = "Batch1"
cfg = project.xps_config(BATCH)
SOURCE = sources.survey_workbook(cfg)
OUT = project.config.resolve(project.config.paths.exports_root) / f"XPS_validation_{BATCH}"
OUT.mkdir(parents=True, exist_ok=True)

# Pick 2–3 spectra that test different things, e.g. overlapping singlets, a weak doublet, a many-component C 1s:
VALIDATE = __VALIDATE__        # [(sample, region), ...]

# Pass criteria (agreement between the two programs)
TOL_BE, TOL_FWHM, TOL_AREA_PCT, TOL_REGION_AREA = 0.10, 0.15, 3.0, 3.0

## 1. Export spectra + this code's results

In [ ]:
if not VALIDATE or SOURCE is None:
    raise SystemExit("Set VALIDATE above (and make sure this batch has an Avantage workbook source).")

results = xf.fit_workbook(SOURCE, cfg, samples=sorted({s for s, _ in VALIDATE}),
                          regions=sorted({r for _, r in VALIDATE}), monte_carlo_runs=100, verbose=False)
picked = [o for o in results if (o.sample, o.region) in VALIDATE]
ours, template = [], []
for o in picked:
    spec = pd.DataFrame({"BE (eV)": o.x[::-1].round(3), "Counts/s": o.y[::-1]})
    spec.to_csv(OUT / f"{o.sample}_{o.region}_corrected.txt", sep="\t", index=False)
    spec.to_excel(OUT / f"{o.sample}_{o.region}_corrected.xlsx", index=False)
    for _, r in o.table.iterrows():
        ours.append({"sample": o.sample, "region": o.region, "component": r["component"],
                     "BE (eV)": r["BE (eV)"], "FWHM (eV)": r["FWHM (eV)"], "area %": round(r["area %"], 2),
                     "± area % (MC)": r.get("± area %", np.nan), "region area": round(o.total_area, 1)})
        template.append({"sample": o.sample, "region": o.region, "component": r["component"],
                         "BE (eV)": "", "FWHM (eV)": "", "area": ""})
ours = pd.DataFrame(ours)
ours.to_csv(OUT / "this_code_results.csv", index=False)
tpl = OUT / "kherve_results.csv"
if not tpl.exists():                      # never overwrite your typed-in values
    pd.DataFrame(template).to_csv(tpl, index=False)
print("Exported to", OUT)
ours

### Settings to use in KherveFitting for each spectrum

In [ ]:
rows = []
for o in picked:
    m = o.model
    for c in m.components:
        rows.append({"spectrum": f"{o.sample} {o.region}", "background": m.background,
                     "bg range (eV)": f"{min(m.fit_range)}–{max(m.fit_range)}", "avg points": m.n_avg,
                     "component": c.name, "start BE": c.center, "BE limits": f"{c.window[0]}–{c.window[1]}",
                     "FWHM limits": f"{c.fwhm_bounds[0]}–{c.fwhm_bounds[1]}",
                     "line shape": f"GL sum / pseudo-Voigt, {int(c.lg*100)}% Lorentzian" if c.shape == "gl" else "LA",
                     "doublet": f"Δ {c.splitting} eV, area ratio {c.ratio}, same FWHM" if c.splitting else ""})
pd.DataFrame(rows)

## 2. Compare (run after filling in `kherve_results.csv`)

In [ ]:
kh = pd.read_csv(OUT / "kherve_results.csv")
for col in ("BE (eV)", "FWHM (eV)", "area"):
    kh[col] = pd.to_numeric(kh[col], errors="coerce")
if kh["area"].isna().all():
    print("kherve_results.csv is still empty — fit the spectra in KherveFitting first (see checklist).")
else:
    kh["area %"] = 100 * kh["area"] / kh.groupby(["sample", "region"])["area"].transform("sum")
    kh["region area"] = kh.groupby(["sample", "region"])["area"].transform("sum")
    cmp = ours.merge(kh, on=["sample", "region", "component"], suffixes=(" code", " Kherve"))
    cmp["Δ BE"] = (cmp["BE (eV) code"] - cmp["BE (eV) Kherve"]).round(3)
    cmp["Δ FWHM"] = (cmp["FWHM (eV) code"] - cmp["FWHM (eV) Kherve"]).round(3)
    cmp["Δ area %"] = (cmp["area % code"] - cmp["area % Kherve"]).round(2)
    major = cmp[["area % code", "area % Kherve"]].max(axis=1) >= 10
    within_mc = cmp["Δ area %"].abs() <= 2 * cmp["± area % (MC)"].fillna(0)
    cmp["BE"] = np.where(cmp["Δ BE"].abs() <= TOL_BE, "PASS", "CHECK")
    cmp["FWHM"] = np.where(cmp["Δ FWHM"].abs() <= TOL_FWHM, "PASS", "CHECK")
    cmp["area"] = np.where(~major, "minor (info)",
                           np.where((cmp["Δ area %"].abs() <= TOL_AREA_PCT) | within_mc, "PASS", "CHECK"))
    region = cmp.groupby(["sample", "region"])[["region area code", "region area Kherve"]].first()
    region["Δ region area %"] = (100 * (region["region area code"] / region["region area Kherve"] - 1)).round(2)
    region["background"] = np.where(region["Δ region area %"].abs() <= TOL_REGION_AREA, "PASS", "CHECK")
    display(cmp[["sample", "region", "component", "BE (eV) code", "BE (eV) Kherve", "Δ BE", "BE",
                 "FWHM (eV) code", "FWHM (eV) Kherve", "Δ FWHM", "FWHM",
                 "area % code", "area % Kherve", "± area % (MC)", "Δ area %", "area"]])
    display(region)
    checks = (cmp[["BE", "FWHM", "area"]] == "CHECK").sum().sum() + (region["background"] == "CHECK").sum()
    print("\nRESULT:", "ALL PASS — the code reproduces KherveFitting on these spectra."
          if checks == 0 else f"{checks} item(s) to CHECK — see 'If something does not match' in the checklist.")
    cmp.to_csv(OUT / "comparison.csv", index=False)